<a href="https://colab.research.google.com/github/shin-noda/leetcode-neetcode-250/blob/main/Problem460.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Brute Force Idea
Store all nodes in the cache.

Manually scan nodes from the tail until use count < current use count
(This is O(N))

Another idea:
Store the first node which has that use counter like
Node 1(use counter: 10), Node 2(use counter: 10), ...
Node a(use counter: 8), Node b(use counter: 8), ...

and we store those Node 1, Node 2, like the leading node of those use counter.
When a new node is called or put, the leading node of one use counter will be replaced.



"""

In [ ]:
class Node:
    def __init__(self, key, val, counter):
        self.key = key
        self.val = val
        self.counter = counter
        self.prev = None
        self.next = None


class LFUCache:

    def __init__(self, capacity):
        self.size = 0
        self.capacity = capacity

        # Map: key -> Node
        self.cache = {}

        # Show the leading node for each occurrenc
        self.leaders = {}

        # Create sentinel nodes
        # Note: we don't count them as the part of the size
        self.head = Node(0, 0, float('inf'))
        self.tail = Node(0, 0, 0)

        # Add head and tail as the leader of occurrence inf and 0
        self.leaders[float('inf')] = self.head
        self.leaders[0] = self.tail

        # Connect head and tail
        # New nodes will always between the head and the tail nodes
        self.head.next = self.tail
        self.tail.prev = self.head


    def _add(self, node, curr_leader):
        # Add to cache if it doesn't exist
        if node.key not in self.cache:
            self.cache[node.key] = node

        # Increment the counter
        node.counter += 1

        prev = curr_leader
        next = prev.next

        # Connect the current node
        node.prev = prev
        prev.next = node

        node.next = next
        next.prev = node

        # Update the current leader
        self.leaders[node.counter] = node


    def _remove(self, node):
        prev = node.prev
        next = node.next

        # Connect the prev and next
        prev.next = next
        next.prev = prev

        # Check if node was the leader of that occurence:
        if node.counter in self.leaders and self.leaders[node.counter] == node:
            if node.counter == next.counter:
                # The next node is the next leader of this counter
                self.leaders[node.counter] = next

            else:
                # This coccurence does not exist, so we remove this key
                self.leaders.pop(node.counter)

        # Remove the node from the cache
        self.cache.pop(node.key)

        # Sever the current node
        node.prev = None
        node.next = None



    ##### This is for debugging
    def _print_all_nodes(self):
        head = self.head

        print("Print All Nodes")

        while head:
            print(str(head.key) + " " + str(head.counter))
            head = head.next

        print("\nPrint All Leaders")
        for key in self.leaders:
            leader = self.leaders[key]
            print(str(leader.counter) + " " + str(leader.key))

        print()


    def get(self, key):
        # Must be O(1) average time complexity
        if key not in self.cache:
            return -1

        node = self.cache[key]
        prev = node.prev

        # node's counter is not incremented yet
        # curr_leader is either next front of the current node or
        # the current leader that will be replaced by node
        # There are 3 possibilities:
        # 1. prev.counter > node.counter like 5 > 3 + 1
        # 2. prev.counter == node.counter + 1 like 5 == 4 + 1
        # 3. prev.counter == node.counter + 1 like 1 > 1 + 1
        curr_leader = None
        # Case 1
        if prev.counter > node.counter + 1:
            # The position doesn't chage
            curr_leader = prev

        # Case 2
        elif prev.counter == node.counter + 1:
            end_prev = self.leaders[prev.counter]

            # Go to the front node of the leader node of the prev
            curr_leader = end_prev.prev

        # Case 3
        else:
            curr_leader = self.leaders[prev.counter].prev

            if curr_leader.counter == node.counter + 1:
                curr_leader = self.leaders[curr_leader.counter].prev

        self._remove(node)
        self._add(node, curr_leader)

        # self._print_all_nodes()

        return node.val


    def put(self, key, value):
        # Must be O(1) average time complexity
        # Check if the key exists or not
        if key in self.cache:
            node = self.cache[key]
            node.val = value
            prev = node.prev

            # node's counter is not incremented yet
            # curr_leader is either next front of the current node or
            # the current leader that will be replaced by node
            # There are 3 possibilities:
            # 1. prev.counter > node.counter like 5 > 3 + 1
            # 2. prev.counter == node.counter + 1 like 5 == 4 + 1
            # 3. prev.counter == node.counter + 1 like 1 > 1 + 1
            curr_leader = None
            # Case 1
            if prev.counter > node.counter + 1:
                # The position doesn't chage
                curr_leader = prev

            # Case 2
            elif prev.counter == node.counter + 1:
                end_prev = self.leaders[prev.counter]

                # Go to the front node of the leader node of the prev
                curr_leader = end_prev.prev

            # Case 3
            else:
                curr_leader = self.leaders[prev.counter].prev

                if curr_leader.counter == node.counter + 1:
                    curr_leader = self.leaders[curr_leader.counter].prev

            self._remove(node)
            self._add(node, curr_leader)

        # The key doesn't exist, so create a new node
        else:
            if self.size >= self.capacity:
                removed_node = self.tail.prev
                self._remove(removed_node)
                self.size -= 1

            node = Node(key, value, 0)

            # node's counter is not incremented yet
            prev = self.tail.prev
            curr_leader = None

            # Check if prev's couter is 1
            if prev.counter > 1:
                curr_leader = prev

            else:
                curr_leader = self.leaders[1].prev

            self._add(node, curr_leader)
            self.size += 1

        # self._print_all_nodes()

In [ ]:
# Your LFUCache object will be instantiated and called as such:
# obj = LFUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)